# Destilar la voz **Alex** (Kokoro) a Piper
Timbre de Alex, sin grabar nada. Guarda en tu Google Drive y **retoma si Colab se corta**.

**Antes:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.
**Celda 1** prepara · **Celda 2** entrena (reejecutable) · **Celda 3** exporta y descarga.

In [ ]:
#@title 1. PREPARAR — anti-desconexión, Google Drive, dataset e instalación
import IPython, torch, os
IPython.display.display(IPython.display.Javascript('function _keep(){ document.querySelector("colab-toolbar-button#connect") && document.querySelector("colab-toolbar-button#connect").click() } setInterval(_keep, 60000)'))
assert torch.cuda.is_available(), "Activá T4: Entorno de ejecución -> Cambiar tipo de entorno -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
WORK = "/content/drive/MyDrive/LoudVox/alex"
os.makedirs(WORK, exist_ok=True)
print("Los checkpoints se guardan en:", WORK)

# --- Generar el dataset con la voz Alex (Kokoro) ---
!pip install -q kokoro-onnx==0.5.0
!wget -q -nc -O /content/kokoro.onnx "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"
!wget -q -nc -O /content/voices.bin "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"
!wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"
import wave, numpy as np
from kokoro_onnx import Kokoro
frases = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]
print(f"{len(frases)} frases. Generando audio con Alex...")
kokoro = Kokoro("/content/kokoro.onnx", "/content/voices.bin")
os.makedirs("/content/dataset/wavs", exist_ok=True)
rows, total = [], 0.0
for i, f in enumerate(frases):
    try: s, r = kokoro.create(f, voice="em_alex", speed=1.0, lang="es")
    except Exception: continue
    idx = np.linspace(0, len(s)-1, int(len(s)*22050/r))
    d = np.clip(np.interp(idx, np.arange(len(s)), s)*32767, -32768, 32767).astype(np.int16)
    if not 1.0 <= len(d)/22050 <= 20.0: continue
    n = f"f{i:05d}.wav"
    with wave.open(f"/content/dataset/wavs/{n}","wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(d.tobytes())
    rows.append(f"{n}|{f}"); total += len(d)/22050
    if len(rows)%200==0: print(f"  {len(rows)} frases...")
open("/content/dataset/metadata.csv","w",encoding="utf-8").write("\n".join(rows)+"\n")
print(f"Dataset: {len(rows)} clips, {total/60:.1f} min.")

print("Instalando Piper (piper1-gpl)...")
!apt-get -q update -y > /dev/null 2>&1
!apt-get -q install -y build-essential cmake ninja-build espeak-ng > /dev/null 2>&1
%cd /content
![ -d piper1-gpl ] || git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip install -q -e '.[train]'
!pip install -q --upgrade scikit-build "protobuf==3.20.3"
!bash build_monotonic_align.sh > /dev/null 2>&1
!python setup.py build_ext --inplace > /dev/null 2>&1
!wget -q -nc -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"
print("\n=== LISTO. Corré la celda 2 para entrenar. ===")

In [ ]:
#@title 2. ENTRENAR — guarda en Drive; si Colab corta, reejecutá esta celda y retoma
import glob, os
WORK = "/content/drive/MyDrive/LoudVox/alex"
prev = sorted(glob.glob(WORK + "/**/*.ckpt", recursive=True), key=os.path.getmtime)
start = prev[-1] if prev else "/content/base.ckpt"
print(("Retomando desde: " if prev else "Empezando del checkpoint español: ") + start)
!cd /content/piper1-gpl && python -m piper.train fit \
  --data.voice_name "alex" \
  --data.csv_path /content/dataset/metadata.csv \
  --data.audio_dir /content/dataset/wavs \
  --data.espeak_voice es \
  --data.cache_dir /content/cache \
  --data.config_path "{WORK}/alex.onnx.json" \
  --data.batch_size 16 \
  --model.sample_rate 22050 \
  --data.validation_split 0 --data.num_test_examples 0 \
  --trainer.default_root_dir "{WORK}" \
  --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.max_epochs 1000 \
  --trainer.precision 16-mixed \
  --checkpoint.every_n_epochs 20 --checkpoint.save_top_k -1 --checkpoint.monitor null \
  --last_checkpoint.every_n_epochs 20 \
  --ckpt_path "{start}"

In [ ]:
#@title 3. EXPORTAR Y DESCARGAR — corré esto cuando quieras (aunque hayas cortado el entrenamiento)
import glob, os
WORK = "/content/drive/MyDrive/LoudVox/alex"
ck = sorted(glob.glob(WORK + "/**/*.ckpt", recursive=True), key=os.path.getmtime)
assert ck, "Todavía no hay checkpoint en Drive. Dejá correr la celda 2 unos minutos (guarda cada 20 epochs)."
print("Exportando desde:", ck[-1])
!cd /content/piper1-gpl && python -m piper.train.export_onnx --checkpoint "{ck[-1]}" --output-file "{WORK}/alex.onnx"
from google.colab import files
files.download(WORK + "/alex.onnx")
files.download(WORK + "/alex.onnx.json")
print("Copiá ambos a tu carpeta de voces de LoudVox y elegí 'alex' en Configuración")